In [0]:
# Optional dependency installer (runs only when explicitly enabled)
AUTO_INSTALL_MISSING_PACKAGES = False
REQUIRED_PACKAGES = [
    "sentence-transformers",
    "transformers",
    "accelerate",
    "mlflow",
    "databricks-sdk",
]

import importlib.metadata as _ilm
import subprocess, sys

missing = []
for pkg in REQUIRED_PACKAGES:
    try:
        _ilm.version(pkg)
    except Exception:
        missing.append(pkg)

if missing:
    print(f"Missing packages: {missing}")
    if AUTO_INSTALL_MISSING_PACKAGES:
        cmd = [sys.executable, "-m", "pip", "install", "-q"] + missing
        print("Installing missing packages:", " ".join(missing))
        subprocess.check_call(cmd)
        print("Installation complete. Re-run this notebook from Cell 1.")
    else:
        print("AUTO_INSTALL_MISSING_PACKAGES=False, so install is skipped.")
else:
    print("All required packages are already available; skipping install.")


All required packages are already available; skipping install.


In [0]:
import os
import re
import math
from datetime import datetime
from typing import Dict, List, Tuple

import numpy as np
from sentence_transformers import SentenceTransformer, CrossEncoder
from transformers import pipeline
from pyspark.sql import functions as F
from pyspark.sql.window import Window


def log(msg: str):
    print(f"[{datetime.now().strftime('%H:%M:%S')}] {msg}")


def s(v):
    return "" if v is None else str(v)


def clean(t: str) -> str:
    return re.sub(r"\s+", " ", s(t)).strip()


def toks(t: str):
    return [x for x in re.findall(r"[a-zA-Z0-9]+", s(t).lower()) if len(x) > 2]


def unique_keep_order(values):
    out = []
    seen = set()
    for v in values:
        k = clean(v).lower()
        if not k or k in seen:
            continue
        seen.add(k)
        out.append(clean(v))
    return out


In [0]:
EMBEDDING_DELTA_PATH = "/Volumes/workspace/legal_data/vector_db_test/legal_embeddings_delta"
EMBEDDING_TABLE_NAME = "workspace.default.legal_embeddings_test"

PRIMARY_EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
FALLBACK_EMBED_MODEL = "sentence-transformers/paraphrase-MiniLM-L3-v2"
RERANK_MODEL = "cross-encoder/ms-marco-MiniLM-L-6-v2"

DATABRICKS_LLM_ENDPOINT = os.environ.get("DATABRICKS_LLM_ENDPOINT", "").strip()
LLM_ENDPOINT_CANDIDATES = [
    DATABRICKS_LLM_ENDPOINT,
    "databricks-meta-llama-3-3-70b-instruct",
    "databricks-meta-llama-3-1-70b-instruct",
    "databricks-mixtral-8x7b-instruct",
]

LOCAL_LLM_MODELS = [
    "google/flan-t5-base",
    "google/flan-t5-small",
]

TOP_K = 8
POOL_K = 48
RERANK_K = 24
MAX_CONTEXT_CHARS = 5000
MIN_RELEVANCE_SCORE = 0.34
MIN_LEXICAL_SCORE = 0.06


In [0]:
def extract_text(resp) -> str:
    if resp is None:
        return ""
    if isinstance(resp, str):
        return resp.strip()
    if isinstance(resp, list) and resp:
        return extract_text(resp[0])
    if isinstance(resp, dict):
        for key in ["generated_text", "text", "output", "answer"]:
            val = resp.get(key)
            if isinstance(val, str) and val.strip():
                return val.strip()
        preds = resp.get("predictions")
        if isinstance(preds, list) and preds:
            return extract_text(preds[0])
        choices = resp.get("choices")
        if isinstance(choices, list) and choices:
            c0 = choices[0]
            if isinstance(c0, dict):
                msg = c0.get("message")
                if isinstance(msg, dict) and isinstance(msg.get("content"), str):
                    return msg["content"].strip()
                if isinstance(c0.get("text"), str):
                    return c0["text"].strip()
    return ""


def load_embedder():
    for name in [PRIMARY_EMBED_MODEL, FALLBACK_EMBED_MODEL]:
        try:
            model = SentenceTransformer(name)
            _ = model.encode(["health check"], show_progress_bar=False)
            log(f"Embedding model ready: {name}")
            return model, name
        except Exception as e:
            log(f"Embedding model failed ({name}): {e}")
    return None, "unavailable"


def load_reranker():
    try:
        rr = CrossEncoder(RERANK_MODEL)
        _ = rr.predict([("hello", "world")])
        return rr, RERANK_MODEL
    except Exception as e:
        log(f"Reranker unavailable: {e}")
        return None, "unavailable"


def load_dbx_client():
    try:
        import mlflow.deployments

        return mlflow.deployments.get_deploy_client("databricks")
    except Exception as e:
        log(f"Databricks endpoint client unavailable: {e}")
        return None


def try_endpoint_once(client, endpoint_name: str, prompt: str) -> Tuple[str, str]:
    try:
        resp = client.predict(
            endpoint=endpoint_name,
            inputs={"messages": [{"role": "user", "content": prompt}], "temperature": 0.0, "max_tokens": 420},
        )
        text = extract_text(resp)
        if text:
            return text, ""
    except Exception as e:
        chat_err = str(e)
    else:
        chat_err = "empty chat response"

    try:
        resp = client.predict(
            endpoint=endpoint_name,
            inputs={"prompt": prompt, "temperature": 0.0, "max_tokens": 420},
        )
        text = extract_text(resp)
        if text:
            return text, ""
        return "", f"empty completion response: {endpoint_name}"
    except Exception as e:
        return "", f"chat_error={chat_err}; completion_error={e}"


def load_local_llm():
    for model_name in LOCAL_LLM_MODELS:
        try:
            llm = pipeline(
                "text2text-generation",
                model=model_name,
                max_new_tokens=420,
                do_sample=False,
                temperature=0.0,
            )
            log(f"Local generation model ready: {model_name}")
            return llm, model_name
        except Exception as e:
            log(f"Local generation model failed ({model_name}): {e}")
    return None, "unavailable"


def infer_source_group(category: str, source: str, act: str) -> str:
    blob = f"{category} {source} {act}".lower()
    checks = [
        ("constitution", "constitution"),
        ("criminal_law", "criminal_law"),
        ("civil_law", "civil_law"),
        ("family_law", "family_law"),
        ("traffic_rules", "traffic_rules"),
        ("judgment", "judgments"),
        ("reports", "reports"),
        ("acts", "acts"),
    ]
    for key, val in checks:
        if key in blob:
            return val
    return "general"


if "_LEXAI06_RUNTIME" in globals() and _LEXAI06_RUNTIME.get("ready", False):
    embedder = _LEXAI06_RUNTIME["embedder"]
    embedder_name = _LEXAI06_RUNTIME["embedder_name"]
    reranker = _LEXAI06_RUNTIME["reranker"]
    reranker_name = _LEXAI06_RUNTIME["reranker_name"]
    dbx = _LEXAI06_RUNTIME["dbx"]
    llm_backend = _LEXAI06_RUNTIME["llm_backend"]
    local_llm = _LEXAI06_RUNTIME["local_llm"]
    records = _LEXAI06_RUNTIME["records"]
    INVERTED_INDEX = _LEXAI06_RUNTIME["INVERTED_INDEX"]
    DOC_FREQ = _LEXAI06_RUNTIME["DOC_FREQ"]
    AVG_DOC_LEN = _LEXAI06_RUNTIME["AVG_DOC_LEN"]
    N_DOCS = _LEXAI06_RUNTIME["N_DOCS"]
    print("Runtime already initialized in this session; reusing cached models and index.")
else:
    embedder, embedder_name = load_embedder()
    if embedder is None:
        raise RuntimeError("No embedding model could be loaded.")

    reranker, reranker_name = load_reranker()
    dbx = load_dbx_client()

    llm_backend = {"type": "none", "name": "", "client": None, "model": None, "errors": []}
    if dbx is not None:
        test_prompt = "Reply with OK"
        for ep in [x for x in LLM_ENDPOINT_CANDIDATES if x]:
            txt, err = try_endpoint_once(dbx, ep, test_prompt)
            if txt:
                llm_backend.update({"type": "endpoint", "name": ep, "client": dbx})
                log(f"Using endpoint backend: {ep}")
                break
            llm_backend["errors"].append(f"{ep}: {err}")

    local_llm, local_llm_name = (None, "unavailable")
    if llm_backend["type"] == "none":
        local_llm, local_llm_name = load_local_llm()
        if local_llm is not None:
            llm_backend.update({"type": "local", "name": local_llm_name, "model": local_llm})

    try:
        emb_df = spark.read.format("delta").load(EMBEDDING_DELTA_PATH)
    except Exception as e:
        log(f"Path load failed, trying table fallback: {e}")
        emb_df = spark.table(EMBEDDING_TABLE_NAME)

    cols = [
        "chunk_id",
        "chunk_text",
        "act_name",
        "section_number",
        "category",
        "file_name",
        "embedding",
        "section_refs",
        "is_penalty_related",
        "is_helmet_related",
    ]
    available = [c for c in cols if c in emb_df.columns]
    required = ["chunk_id", "chunk_text", "act_name", "section_number", "category", "file_name", "embedding"]
    miss = [c for c in required if c not in emb_df.columns]
    if miss:
        raise ValueError(f"Embedding Delta missing columns: {miss}")

    if "updated_at" not in emb_df.columns:
        emb_df = emb_df.withColumn("updated_at", F.current_timestamp())

    w = Window.partitionBy("chunk_id").orderBy(F.col("updated_at").desc_nulls_last())
    latest_df = (
        emb_df.withColumn("rn", F.row_number().over(w))
        .filter(F.col("rn") == 1)
        .drop("rn")
        .dropna(subset=["chunk_id", "chunk_text", "embedding"])
    )

    records = []
    for ridx, r in enumerate(latest_df.select(*available).toLocalIterator()):
        refs = []
        if "section_refs" in available and getattr(r, "section_refs", None):
            refs = [x.strip().upper() for x in s(r.section_refs).split(",") if x.strip()]

        text = clean(r.chunk_text)
        tok_list = toks(text)
        tf = {}
        for t in tok_list:
            tf[t] = tf.get(t, 0) + 1

        source_group = infer_source_group(s(getattr(r, "category", "")), s(getattr(r, "file_name", "")), s(getattr(r, "act_name", "")))

        records.append(
            {
                "idx": ridx,
                "id": s(r.chunk_id),
                "text": text,
                "text_norm": text.lower(),
                "act": s(r.act_name),
                "act_norm": s(r.act_name).lower(),
                "section": s(r.section_number),
                "section_norm": s(r.section_number).upper(),
                "category": s(r.category),
                "source": s(r.file_name),
                "source_group": source_group,
                "emb": np.array([float(x) for x in r.embedding], dtype=np.float32),
                "tokens": set(tok_list),
                "token_freq": tf,
                "doc_len": len(tok_list),
                "section_refs": refs,
                "is_penalty_related": int(getattr(r, "is_penalty_related", 0) or 0),
                "is_helmet_related": int(getattr(r, "is_helmet_related", 0) or 0),
            }
        )

    if not records:
        raise RuntimeError("No records loaded from latest embeddings snapshot")

    INVERTED_INDEX = {}
    DOC_FREQ = {}
    for rec in records:
        for t in rec["token_freq"].keys():
            DOC_FREQ[t] = DOC_FREQ.get(t, 0) + 1
            INVERTED_INDEX.setdefault(t, []).append(rec["idx"])

    N_DOCS = len(records)
    AVG_DOC_LEN = sum(max(1, r["doc_len"]) for r in records) / max(1, N_DOCS)

    _LEXAI06_RUNTIME = {
        "ready": True,
        "embedder": embedder,
        "embedder_name": embedder_name,
        "reranker": reranker,
        "reranker_name": reranker_name,
        "dbx": dbx,
        "llm_backend": llm_backend,
        "local_llm": local_llm,
        "records": records,
        "INVERTED_INDEX": INVERTED_INDEX,
        "DOC_FREQ": DOC_FREQ,
        "AVG_DOC_LEN": AVG_DOC_LEN,
        "N_DOCS": N_DOCS,
    }

print("--- Runtime Status ---")
print("Records loaded:", len(records))
print("Embedding dim:", len(records[0]["emb"]))
print("Avg doc len:", round(float(AVG_DOC_LEN), 2))
print("Embedder:", embedder_name)
print("Reranker:", reranker_name)
print("LLM backend:", f"{llm_backend['type']} ({llm_backend['name']})" if llm_backend["type"] != "none" else "none")
if llm_backend.get("errors"):
    print("LLM backend diagnostics:")
    for e in llm_backend["errors"][:5]:
        print(" -", e)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[20:04:19] Embedding model ready: sentence-transformers/all-MiniLM-L6-v2


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[20:04:22] Using endpoint backend: databricks-meta-llama-3-3-70b-instruct
--- Runtime Status ---
Records loaded: 5194
Embedding dim: 384
Embedder: sentence-transformers/all-MiniLM-L6-v2
Reranker: cross-encoder/ms-marco-MiniLM-L-6-v2
LLM backend: endpoint (databricks-meta-llama-3-3-70b-instruct)


In [0]:
def cos(a: np.ndarray, b: np.ndarray) -> float:
    return float(np.dot(a, b) / ((float(np.linalg.norm(a)) + 1e-12) * (float(np.linalg.norm(b)) + 1e-12)))


def parse_query_sections(query: str) -> List[str]:
    refs = []
    for m in re.findall(r"(?:section|sec\.?|article|art\.?)\s*(\d+[A-Za-z-]*)", query, flags=re.IGNORECASE):
        refs.append(m.upper())
    return unique_keep_order(refs)


def detect_domain(query: str) -> str:
    q = query.lower()
    if any(x in q for x in ["helmet", "traffic", "vehicle", "challan", "headgear", "driving", "speeding", "drunk driving"]):
        return "traffic"
    if any(x in q for x in ["murder", "homicide", "crime", "imprisonment", "punishment", "bns", "ipc", "bail"]):
        return "criminal"
    if any(x in q for x in ["bond", "employment", "salary", "resign", "contract", "notice period", "company"]):
        return "employment"
    if any(x in q for x in ["marriage", "divorce", "succession", "dowry", "domestic violence"]):
        return "family"
    return "general"


def generate_query_variants(query: str) -> List[str]:
    q = clean(query)
    ql = q.lower()
    variants = [q]

    if "helmet" in ql or "headgear" in ql:
        variants.append("helmet penalty motor vehicles act section 129 section 177 section 194D")
        variants.append("protective headgear legal penalty two wheeler india")

    if "section 129" in ql or re.search(r"\b129\b", ql):
        variants.append("what does section 129 motor vehicles act require")

    if any(x in ql for x in ["murder", "homicide"]):
        variants.append("punishment for murder bns ipc equivalent section")

    if "bond" in ql and any(x in ql for x in ["company", "employment", "resign"]):
        variants.append("employment bond breach legal consequences india contract act section 73 section 74")

    if "mobile" in ql and "driving" in ql:
        variants.append("mobile phone while driving offence motor vehicles act")

    variants.append(" ".join(toks(q)[:14]))
    return unique_keep_order([v for v in variants if clean(v)])


def source_group_bonus(domain: str, query: str, source_group: str) -> float:
    q = query.lower()
    case_law_query = any(x in q for x in ["judgment", "precedent", "case law", "supreme court", "high court"])
    bonus = 0.0

    if case_law_query:
        if source_group == "judgments":
            bonus += 0.20
        elif source_group == "reports":
            bonus += 0.04
    else:
        if source_group in {"acts", "constitution", "criminal_law", "civil_law", "family_law", "traffic_rules"}:
            bonus += 0.14
        elif source_group == "judgments":
            bonus -= 0.03
        elif source_group == "reports":
            bonus -= 0.08

    if domain == "traffic" and source_group == "traffic_rules":
        bonus += 0.14
    if domain == "criminal" and source_group == "criminal_law":
        bonus += 0.14
    if domain == "employment" and source_group in {"civil_law", "acts"}:
        bonus += 0.12
    if domain == "family" and source_group == "family_law":
        bonus += 0.14

    return bonus


def bm25_score(query_terms: List[str], rec: Dict, k1: float = 1.5, b: float = 0.75) -> float:
    score = 0.0
    dl = max(1, rec.get("doc_len", 1))
    for t in query_terms:
        tf = rec["token_freq"].get(t, 0)
        if tf <= 0:
            continue
        df = DOC_FREQ.get(t, 0)
        idf = math.log(1 + ((N_DOCS - df + 0.5) / (df + 0.5))) if N_DOCS > 0 else 0.0
        denom = tf + k1 * (1 - b + b * (dl / max(1.0, AVG_DOC_LEN)))
        score += idf * ((tf * (k1 + 1)) / max(1e-9, denom))
    return float(score)


def candidate_ids_from_inverted(query_terms: List[str], hard_cap: int = 15000) -> List[int]:
    ids = set()
    for t in query_terms:
        postings = INVERTED_INDEX.get(t, [])
        if postings:
            ids.update(postings[:5000])
        if len(ids) >= hard_cap:
            break
    if not ids:
        return list(range(len(records)))
    return list(ids)


def filter_records(query: str, rows: List[Dict], domain: str):
    if not rows:
        return []
    if domain == "general":
        return rows

    strict = []
    soft = []
    q_terms = set(toks(query))
    for r in rows:
        act = r["act_norm"]
        txt = r["text_norm"]
        overlap = sum(1 for t in q_terms if t in r["tokens"])

        if domain == "traffic":
            if ("motor vehicle" in act or "traffic" in act or r["source_group"] == "traffic_rules") and any(
                k in txt for k in ["helmet", "headgear", "motor cycle", "motorcycle", "two-wheeler", "driving", "licence"]
            ):
                strict.append(r)
            elif overlap >= 2:
                soft.append(r)
        elif domain == "criminal":
            if any(k in txt for k in ["murder", "homicide", "imprisonment", "punishable", "offence"]):
                strict.append(r)
            elif overlap >= 2:
                soft.append(r)
        elif domain == "employment":
            if any(k in txt for k in ["contract", "bond", "agreement", "damages", "specific relief", "notice"]):
                strict.append(r)
            elif overlap >= 2:
                soft.append(r)
        elif domain == "family":
            if any(k in txt for k in ["marriage", "divorce", "maintenance", "dowry", "custody", "succession"]):
                strict.append(r)
            elif overlap >= 2:
                soft.append(r)
        else:
            if overlap >= 2:
                soft.append(r)

    return strict + soft if strict else soft if soft else rows


def score_record(query: str, rec: Dict, q_vec, q_terms: List[str], q_sections: set, domain: str):
    txt = rec["text_norm"]
    dense = cos(q_vec, rec["emb"]) if q_vec is not None else 0.0
    lex = sum(1 for t in set(q_terms) if t in rec["tokens"]) / max(1, len(set(q_terms)))
    bm25_raw = bm25_score(q_terms, rec)

    meta = 0.0
    if rec.get("is_penalty_related", 0) == 1:
        meta += 0.07
    if any(x in txt for x in ["penalty", "fine", "punishable", "challan", "imprisonment", "liable"]):
        meta += 0.08

    if q_sections:
        if rec["section_norm"] in q_sections:
            meta += 0.22
        if set(rec.get("section_refs", [])).intersection(q_sections):
            meta += 0.16

    meta += source_group_bonus(domain, query, rec.get("source_group", "general"))

    return {
        "dense": float(dense),
        "lex": float(lex),
        "bm25_raw": float(bm25_raw),
        "meta": float(meta),
        "rec": rec,
    }


def retrieve_single_query(query: str, top_n: int = POOL_K):
    domain = detect_domain(query)
    q_terms = toks(query)
    q_sections = set(parse_query_sections(query))
    q_vec = np.array(embedder.encode([query], show_progress_bar=False)[0], dtype=np.float32) if embedder is not None else None

    cand_ids = candidate_ids_from_inverted(q_terms, hard_cap=18000)
    candidates = [records[i] for i in cand_ids]
    candidates = filter_records(query, candidates, domain)

    raw = [score_record(query, r, q_vec, q_terms, q_sections, domain) for r in candidates]
    if not raw:
        return []

    max_bm25 = max(x["bm25_raw"] for x in raw) if raw else 1.0
    scored = []
    for x in raw:
        bm25_norm = x["bm25_raw"] / max(1e-9, max_bm25)
        if q_vec is None:
            score = (0.52 * x["lex"]) + (0.36 * bm25_norm) + x["meta"]
        else:
            score = (0.50 * x["dense"]) + (0.18 * x["lex"]) + (0.22 * bm25_norm) + x["meta"]
        scored.append((float(score), float(x["dense"]), float(max(x["lex"], bm25_norm)), x["rec"]))

    scored.sort(key=lambda z: z[0], reverse=True)
    return scored[:top_n]


def reciprocal_rank_fusion(result_sets: List[List[Tuple]], k: int = 60):
    fused = {}
    for result in result_sets:
        for rank, item in enumerate(result, start=1):
            rid = item[3]["id"]
            base = fused.get(rid, {"rrf": 0.0, "best": item})
            base["rrf"] += 1.0 / (k + rank)
            if item[0] > base["best"][0]:
                base["best"] = item
            fused[rid] = base

    rows = []
    for _, data in fused.items():
        best = data["best"]
        merged_score = (0.76 * best[0]) + (0.24 * data["rrf"])
        rows.append((float(merged_score), best[1], best[2], best[3]))

    rows.sort(key=lambda x: x[0], reverse=True)
    return rows


def hybrid_retrieve(query: str, top_k: int = TOP_K):
    variants = generate_query_variants(query)
    raw_sets = [retrieve_single_query(v, top_n=POOL_K) for v in variants]
    fused = reciprocal_rank_fusion(raw_sets, k=60)

    if reranker is not None and fused:
        pairs = [(query, x[3]["text"][:1000]) for x in fused[:RERANK_K]]
        try:
            ce = reranker.predict(pairs)
            reranked = []
            for i, c in enumerate(ce):
                ce_norm = 1 / (1 + math.exp(-float(c)))
                merged = (0.64 * fused[i][0]) + (0.36 * ce_norm)
                reranked.append((float(merged), fused[i][1], fused[i][2], fused[i][3]))
            reranked.sort(key=lambda x: x[0], reverse=True)
            fused = reranked + fused[RERANK_K:]
        except Exception as e:
            log(f"Reranker skipped: {e}")

    return fused[:top_k], variants


In [0]:
def section_refs(text: str, sec_meta: str = ""):
    refs = []
    meta = s(sec_meta).strip()
    if meta and not meta.lower().startswith("chapter"):
        refs += re.findall(r"\d+[A-Za-z-]*", meta)

    refs += re.findall(r"(?:section|sec\.?|article|art\.?)\s*(\d+[A-Za-z-]*)", s(text), flags=re.IGNORECASE)
    refs = [s(x).upper() for x in refs]
    return unique_keep_order(refs)


def parse_response_preferences(query: str) -> Dict:
    q = query.lower()
    style = "normal"
    if any(k in q for k in ["very short", "one line", "single line"]):
        style = "very_short"
    elif any(k in q for k in ["in short", "briefly", "brief"]):
        style = "short"
    elif any(k in q for k in ["in detail", "in details", "in depth", "detailed", "comprehensive"]):
        style = "detailed"

    word_limit = None
    m = re.search(r"(?:within|under|max(?:imum)?|limit(?:ed)? to)\s*(\d{2,4})\s*(?:words?|works?)", q)
    if m:
        word_limit = int(m.group(1))

    defaults = {"very_short": 45, "short": 85, "normal": 150, "detailed": 260}
    target_words = defaults.get(style, 150)
    if word_limit is not None:
        target_words = max(35, min(word_limit, 380))

    return {"style": style, "word_limit": word_limit, "target_words": target_words}


def apply_word_limit(text: str, target_words: int) -> str:
    words = s(text).split()
    if target_words is None or target_words <= 0 or len(words) <= target_words:
        return s(text).strip()
    trimmed = " ".join(words[:target_words]).strip()
    if not trimmed.endswith((".", "!", "?")):
        trimmed += "."
    return trimmed


def build_context(query: str, ranked):
    terms = toks(query)
    ctx = []
    sections = []
    evidence = []

    for score, vec, lex, r in ranked:
        txt = r["text"]
        low = txt.lower()
        pos = [low.find(t) for t in terms if t in low]
        if pos:
            i = min(pos)
            snippet = txt[max(0, i - 170) : min(len(txt), i + 700)]
        else:
            snippet = txt[:700]

        snippet = clean(snippet)
        if not snippet:
            continue

        sections.extend(section_refs(snippet, r["section"]))
        ctx.append(f"[Act: {r['act']}] [Section: {r['section']}] {snippet}")
        evidence.append(
            {
                "act": r["act"],
                "section": r["section"],
                "score": round(float(score), 4),
                "vector": round(float(vec), 4),
                "lexical": round(float(lex), 4),
                "source": r["source"],
                "source_group": r.get("source_group", "general"),
                "snippet": snippet[:340],
            }
        )

    dedup = unique_keep_order([s(x).upper() for x in sections])[:12]
    return ctx, dedup, evidence


def confidence_gate(ranked, query: str, evidence: List[Dict]):
    if not ranked:
        return False, "no_candidates"

    top_score = float(ranked[0][0])
    avg_lex = sum(float(x[2]) for x in ranked[:3]) / max(1, min(3, len(ranked)))
    q_terms = set(toks(query))
    top_text = " ".join(x[3]["text_norm"][:700] for x in ranked[:3])
    coverage = (sum(1 for t in q_terms if t in top_text) / max(1, len(q_terms))) if q_terms else 0.0

    normative_groups = {"acts", "constitution", "criminal_law", "civil_law", "family_law", "traffic_rules"}
    normative_hits = sum(1 for e in evidence[:5] if e.get("source_group") in normative_groups)
    case_law_query = any(x in query.lower() for x in ["judgment", "precedent", "case law", "supreme court", "high court"])

    ok = (top_score >= MIN_RELEVANCE_SCORE and avg_lex >= MIN_LEXICAL_SCORE) or coverage >= 0.42
    if not case_law_query and normative_hits == 0:
        ok = False

    return ok, f"top_score={top_score:.3f}, avg_lex={avg_lex:.3f}, coverage={coverage:.3f}, normative_hits={normative_hits}"


def build_citations(sections: List[str], evidence: List[Dict]) -> List[str]:
    citations = []
    for e in evidence[:8]:
        act = clean(e.get("act", ""))
        sec = clean(e.get("section", ""))
        if act or sec:
            citations.append(f"{act} - Section {sec}".strip())
    for s0 in sections[:6]:
        citations.append(f"Section {s0}")
    return unique_keep_order(citations)[:10]


In [0]:
def endpoint_generate(prompt: str) -> str:
    if llm_backend.get("type") != "endpoint":
        return ""

    client = llm_backend.get("client")
    endpoint = llm_backend.get("name")
    if client is None or not endpoint:
        return ""

    text, err = try_endpoint_once(client, endpoint, prompt)
    if text:
        return text

    llm_backend.setdefault("errors", []).append(f"Endpoint generation failed: {err}")
    return ""


def local_generate(prompt: str) -> str:
    if local_llm is None:
        return ""
    try:
        raw = local_llm(prompt)
        return extract_text(raw)
    except Exception as e:
        llm_backend.setdefault("errors", []).append(f"Local generation failed: {e}")
        return ""


def domain_advice_line(domain: str) -> str:
    if domain == "traffic":
        return "Follow state traffic challan notifications and keep licence/vehicle documents valid."
    if domain == "criminal":
        return "Seek licensed legal counsel immediately before making any statement to authorities."
    if domain == "employment":
        return "Review contract clauses and send written communication before taking action."
    if domain == "family":
        return "Consult family court or legal aid services with all relevant documents."
    return "Review the cited statutory sections directly before relying on this summary."


def synthesize_from_evidence(query: str, evidence: List[Dict], sections: List[str], target_words: int) -> str:
    domain = detect_domain(query)
    top = evidence[:4]
    law_bits = []
    penalty_bits = []
    for e in top:
        txt = clean(e.get("snippet", ""))
        if txt:
            law_bits.append(txt[:190])
        if any(k in txt.lower() for k in ["penalty", "fine", "punishable", "imprisonment", "liable", "offence"]):
            penalty_bits.append(txt[:190])

    law_line = " ".join(law_bits)[:420] if law_bits else "Relevant legal provisions were retrieved from indexed acts and sections."
    penalty_line = (
        " ".join(penalty_bits)[:320]
        if penalty_bits
        else "Penalty depends on exact section wording and applicable enforcement rules in the cited legal text."
    )

    text = f'''Law:
{law_line}

Penalty:
{penalty_line}

Why this rule exists:
Legal provisions define enforceable obligations, rights, and consequences.

Advice:
{domain_advice_line(domain)}'''
    return apply_word_limit(text, target_words)


def build_generation_prompt(query: str, draft_answer: str, sections: List[str], preferences: Dict, citations: List[str], variants: List[str]) -> str:
    section_text = ", ".join(sections) if sections else "Not clearly identified"
    citation_text = " | ".join(citations[:8]) if citations else "No citations detected"
    variant_text = " | ".join(variants[:4])
    style = preferences.get("style", "normal")
    target_words = preferences.get("target_words", 150)
    style_hint = {
        "very_short": "Keep it very concise.",
        "short": "Keep it concise and direct.",
        "normal": "Keep it clear with moderate detail.",
        "detailed": "Provide detailed but structured explanation.",
    }.get(style, "Keep it clear with moderate detail.")

    return f'''
You are an expert Indian legal assistant.
Rewrite the draft answer for clarity and structure.
Do NOT add any new facts, sections, penalties, or acts beyond the draft/citations.
{style_hint}
Keep output near {target_words} words.

Return exactly this structure:
Law:
...

Penalty:
...

Why this rule exists:
...

Advice:
...

Question:
{query}

Query Variants Used:
{variant_text}

Relevant Sections:
{section_text}

Citations:
{citation_text}

Draft Answer:
{draft_answer}
'''


def ensure_structured_answer(text: str, target_words: int) -> str:
    raw = s(text).strip()
    if not raw:
        return ""
    required_headers = ["Law:", "Penalty:", "Why this rule exists:", "Advice:"]
    if all(h.lower() in raw.lower() for h in required_headers):
        return apply_word_limit(raw, target_words)
    norm = apply_word_limit(clean(raw), target_words)
    return f'''Law:
{norm}

Penalty:
Refer to cited sections for exact penalty.

Why this rule exists:
Legal provisions ensure compliance and public safety.

Advice:
Review cited statutory text before relying on this answer.'''


In [0]:
def high_precision_answer(query: str):
    preferences = parse_response_preferences(query)
    ranked, variants = hybrid_retrieve(query)
    context, sections, evidence = build_context(query, ranked)
    citations = build_citations(sections, evidence)
    is_confident, confidence_reason = confidence_gate(ranked, query, evidence)

    ql = query.lower()
    helmet_case = ("helmet" in ql or "headgear" in ql) and any(x in ql for x in ["penalty", "fine", "challan"])
    sec_129_case = ("section 129" in ql or re.search(r"\b129\b", ql) is not None) and any(
        x in ql for x in ["what", "say", "explain", "meaning", "detail", "short"]
    )

    if helmet_case:
        sec_up = {x.upper() for x in sections}
        if "129" not in sec_up:
            sections.append("129")
        if "177" not in sec_up and "194D" not in sec_up:
            sections.append("177")

        answer = '''Law:
Section 129 of the Motor Vehicles Act requires riders to wear protective headgear while riding two-wheelers in public places.

Penalty:
Violation may attract enforcement under Section 177/194D-style traffic penalty provisions, often including monetary fine and possible licence-related consequences depending on state notifications.

Why this rule exists:
Helmet compliance reduces severe head injuries and road fatalities.

Advice:
Use a BIS-approved helmet with strap fastened and follow challan rules notified by your state traffic authority.'''

        return {
            "answer": apply_word_limit(answer, preferences.get("target_words", 150)),
            "sections": unique_keep_order(sections),
            "citations": unique_keep_order(citations + ["Motor Vehicles Act - Section 129", "Motor Vehicles Act - Section 177", "Motor Vehicles Act - Section 194D"]),
            "mode": "rule_based",
            "source": "traffic_rules",
            "confidence": confidence_reason,
            "preferences": preferences,
            "evidence": evidence[:5],
            "query_variants": variants,
        }

    if sec_129_case:
        sec_up = {x.upper() for x in sections}
        if "129" not in sec_up:
            sections.append("129")
        answer = '''Law:
Section 129 mandates protective headgear for motorcycle riders in public places.

Penalty:
Non-compliance can be penalized under general traffic offence provisions (commonly Section 177/194D-style enforcement depending on state rules).

Why this rule exists:
The provision aims to prevent fatal and life-altering head injuries.

Advice:
Wear a compliant helmet for every ride, including short city commutes.'''
        return {
            "answer": apply_word_limit(answer, preferences.get("target_words", 150)),
            "sections": unique_keep_order(sections),
            "citations": unique_keep_order(citations + ["Motor Vehicles Act - Section 129"]),
            "mode": "rule_based",
            "source": "section_129_rule",
            "confidence": confidence_reason,
            "preferences": preferences,
            "evidence": evidence[:5],
            "query_variants": variants,
        }

    if not context or not is_confident:
        answer = '''Law:
No sufficiently relevant legal context was found in the indexed dataset for this question.

Penalty:
Not available from current retrieval evidence.

Why this rule exists:
Confidence gate prevents inaccurate or hallucinated legal advice.

Advice:
Ask a narrower question with act/section details or expand the indexed legal corpus.'''
        return {
            "answer": apply_word_limit(answer, preferences.get("target_words", 150)),
            "sections": unique_keep_order(sections),
            "citations": citations,
            "mode": "none",
            "source": "low_confidence",
            "confidence": confidence_reason,
            "preferences": preferences,
            "evidence": evidence[:5],
            "query_variants": variants,
        }

    # First build deterministic grounded answer from evidence.
    grounded = synthesize_from_evidence(query, evidence, sections, preferences.get("target_words", 150))

    # Optional polish step: rewrite for readability without adding new legal facts.
    prompt = build_generation_prompt(query, grounded, sections, preferences, citations, variants)
    polished = endpoint_generate(prompt)
    mode = "endpoint_polish"
    if not polished:
        polished = local_generate(prompt)
        mode = "local_polish"

    final_text = ensure_structured_answer(polished, preferences.get("target_words", 150)) if polished else grounded
    if not final_text:
        final_text = grounded
        mode = "evidence_synthesis"

    return {
        "answer": final_text,
        "sections": unique_keep_order(sections),
        "citations": citations,
        "mode": mode if polished else "evidence_synthesis",
        "source": "hybrid_retrieve_rrf_rerank",
        "confidence": confidence_reason,
        "preferences": preferences,
        "evidence": evidence[:5],
        "query_variants": variants,
    }


In [0]:
def format_answer(payload: Dict, render_mode: str = "text"):
    sec = payload.get("sections", [])
    sec_text = ", ".join(sec) if sec else "Refer to applicable legal provisions"
    citations = payload.get("citations", [])
    citation_text = " | ".join(citations[:6]) if citations else "Not available"
    variants = payload.get("query_variants", [])
    variant_text = " | ".join(variants[:4]) if variants else "original_query_only"

    evidence = payload.get("evidence", [])
    evidence_text = " | ".join(
        [f"{e.get('act','')} Sec {e.get('section','')} score={e.get('score','')}" for e in evidence[:3]]
    ) if evidence else "Not available"

    answer_body = s(payload.get("answer", "")).strip()
    confidence = payload.get("confidence", "na")
    prefs = payload.get("preferences", {})

    if render_mode == "html":
        esc = lambda x: (
            s(x).replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;").replace("\n", "<br>")
        )
        return f"""
<div style="font-family:Segoe UI,Arial,sans-serif;line-height:1.45;padding:18px;border:1px solid #d0d7de;border-radius:12px;background:#f7fbff;">
  <div style="font-size:21px;font-weight:700;margin-bottom:8px;">&#9878; High-Precision Legal Explanation</div>
  <div style="font-size:14px;">{esc(answer_body)}</div>
  <hr style="margin:12px 0;border:none;border-top:1px solid #e5e7eb;"/>
  <div><b>&#128214; Relevant Legal Sections:</b> {esc(sec_text)}</div>
  <div><b>&#128278; Citations:</b> {esc(citation_text)}</div>
  <div><b>&#129517; Retrieval Source:</b> {esc(payload.get("source", "unknown"))}</div>
  <div><b>&#128161; Query Variants:</b> {esc(variant_text)}</div>
  <div><b>&#128221; Top Evidence:</b> {esc(evidence_text)}</div>
  <div><b>&#127919; Response Style:</b> {esc(prefs.get("style", "normal"))}, target_words={esc(prefs.get("target_words", "na"))}</div>
  <div><b>&#128200; Confidence:</b> {esc(confidence)}</div>
  <div style="margin-top:10px;color:#555;"><b>&#9888; Disclaimer:</b> AI-generated legal information and not a substitute for professional legal advice.</div>
</div>
"""

    return f"""
[HIGH-PRECISION LEGAL EXPLANATION]

{answer_body}

[RELEVANT LEGAL SECTIONS]
{sec_text}

[CITATIONS]
{citation_text}

[QUERY VARIANTS USED]
{variant_text}

[TOP EVIDENCE]
{evidence_text}

[RETRIEVAL SOURCE]
{payload.get("source", "unknown")}

[RESPONSE STYLE]
style={prefs.get("style", "normal")}, target_words={prefs.get("target_words", "na")}

[CONFIDENCE]
{confidence}

[DISCLAIMER]
This response is AI-generated legal information and not a substitute for professional legal advice.
"""


def show_answer(payload: Dict, prefer_html: bool = True):
    if prefer_html:
        try:
            displayHTML(format_answer(payload, render_mode="html"))
            return
        except Exception:
            pass
    print(format_answer(payload, render_mode="text"))


In [0]:
RUN_06_DEMO = False

if RUN_06_DEMO:
    show_answer(high_precision_answer("Penalty for not wearing helmet in India in short within 120 words"), prefer_html=True)
else:
    print("RUN_06_DEMO=False -> skipped demo query cell.")


⚖ High-Precision Legal Explanation 
 Law: Section 129 of the Motor Vehicles Act requires riders to wear protective headgear while riding two-wheelers in public places. Penalty: Violation may attract enforcement under Section 177/194D-style traffic penalty provisions, often including monetary fine and possible licence-related consequences depending on state notifications. Why this rule exists: Helmet compliance reduces severe head injuries and road fatalities. Advice: Use a BIS-approved helmet with strap fastened and follow challan rules notified by your state traffic authority. 
 
 📖 Relevant Legal Sections: 129, 177, 178, 179, 180, 181, 182, 183, 184, 186, 189, 190 
 🔖 Citations: Motor Vehicles Act 1988 - Section Chapter V | Central Motor Vehicle Rules 1989 - Section Chapter VI | Motor Vehicles Act 1988 - Section Chapter VII | Motor Vehicle Ammendment Act 2019 - Section Chapter XI | Central Motor Vehicle Rules 1989 - Section Chapter I | Motor Vehicles Act 1988 - Section Chapter II 
 🧭 Retrieval Source: traffic_rules 
 💡 Query Variants: Penalty for not wearing helmet in India in short within 120 words | helmet penalty motor vehicles act section 129 section 177 section 194D | protective headgear legal penalty two wheeler india | penalty for not wearing helmet india short within 120 words 
 🎯 Response Style: short, target_words=120 
 📈 Confidence: top_score=0.622, avg_lex=0.487, coverage=0.400 
 ⚠ Disclaimer: AI-generated legal information and not a substitute for professional legal advice.

In [0]:
EVAL_SET = [
    {"query": "penalty for not wearing helmet in very short", "must": {"129"}, "optional": {"177", "194D"}},
    {"query": "what does section 129 say in detail within 200 words", "must": {"129"}, "optional": {"177", "194D"}},
    {"query": "triple riding fine within 100 words", "must": set(), "optional": {"128", "177", "194D"}},
    {"query": "if i leave company bond early what legal consequence in detail", "must": set(), "optional": {"73", "74", "contract"}},
]


In [0]:
def evaluate(eval_set):
    rows = []
    for item in eval_set:
        q = item["query"]
        out = high_precision_answer(q)
        sec = {x.upper() for x in out.get("sections", [])}

        must = {x.upper() for x in item.get("must", set())}
        optional = {x.upper() for x in item.get("optional", set())}

        rows.append(
            {
                "query": q,
                "mode": out.get("mode"),
                "must_hit": must.issubset(sec),
                "optional_hit": bool(optional.intersection(sec)) if optional else True,
                "sections": sorted(list(sec)),
                "confidence": out.get("confidence"),
                "style": out.get("preferences", {}).get("style"),
                "target_words": out.get("preferences", {}).get("target_words"),
                "source": out.get("source"),
            }
        )

    df = spark.createDataFrame(rows)
    df.show(truncate=False)
    df.groupBy("must_hit", "optional_hit", "mode", "source").count().show()
    return df


RUN_06_EVAL = False
if RUN_06_EVAL:
    eval_df = evaluate(EVAL_SET)
else:
    print("RUN_06_EVAL=False -> skipped evaluation cell.")


+----------------------------------------------+----------+--------+------------+--------------------------------------------------------------+------------------------------------------------------------+--------------------------+----------+------------+
|confidence                                    |mode      |must_hit|optional_hit|query                                                         |sections                                                    |source                    |style     |target_words|
+----------------------------------------------+----------+--------+------------+--------------------------------------------------------------+------------------------------------------------------------+--------------------------+----------+------------+
|top_score=0.622, avg_lex=0.487, coverage=0.429|rule_based|true    |true        |penalty for not wearing helmet in very short                  |[129, 177, 178, 179, 180, 181, 182, 183, 184, 186, 189, 190]|traffic_rules           

In [0]:
ENABLE_FINE_TUNING = False

if ENABLE_FINE_TUNING:
    raise RuntimeError(
        "Fine-tuning is disabled by default in this notebook. "
        "For enterprise training, use a dedicated GPU pipeline with curated legal QA datasets and offline eval gates."
    )
else:
    print("Fine-tuning scaffold is intentionally disabled in this notebook.")


Fine-tuning scaffold is intentionally disabled in this notebook.
